# El registro de experimentos de GrantMatch

Una corrida de MLflow por cada modelo de la Semana 10 (línea base, regresión logística, ensamblajes y red neuronal), en el experimento `grantmatch`. Cada corrida registra los hiperparámetros, cómo se dividieron los datos (el año de corte), la huella del archivo de datos, la AUC en entrenamiento y en prueba, los segundos y el pipeline entrenado. Al final marco la corrida final (`candidato` = `final`), cargo su modelo desde el registro y compruebo que reproduce su AUC.

Antes de hacer commit: borra las corridas de prueba del experimento en la interfaz de MLflow (puerto 5050), **Restart** y **Run All**. La tabla sale en `corridas_proyecto.csv`.

In [ ]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from pipeline_proyecto import RUTA_DATOS, anio_de_corte, cargar_datos, cargar_y_dividir, construir_preparacion

logging.getLogger("mlflow").setLevel(logging.ERROR)

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
CORTE = anio_de_corte(cargar_datos())
print("Ano de corte:", CORTE, "  Entrenamiento:", X_ent.shape, "  Prueba:", X_prueba.shape)


def huella(ruta):
    """Los primeros 12 caracteres del MD5 del archivo de datos."""
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
print("Huella de pares_proyecto.csv:", huella_datos)
print("Registro de MLflow:", mlflow.get_tracking_uri())
EXPERIMENTO = "grantmatch"
experimento_de_hoy = mlflow.set_experiment(EXPERIMENTO)

## Entrenar y registrar

In [ ]:
def registrar(nombre, modelo):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params({
            **modelo.get_params(),
            "division": "por fecha",
            "anio_de_corte": CORTE,
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
        })
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        auc_ent = roc_auc_score(y_ent, pipeline.predict_proba(X_ent)[:, 1])
        auc_prueba = roc_auc_score(y_prueba, pipeline.predict_proba(X_prueba)[:, 1])
        mlflow.log_metrics({"auc_entrenamiento": auc_ent, "auc_prueba": auc_prueba, "segundos": segundos})
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre:22s} AUC entrenamiento {auc_ent:.4f}   prueba {auc_prueba:.4f}   {segundos:6.2f} s")

In [ ]:
MODELOS = {
    "linea base": lambda: DummyClassifier(strategy="prior"),
    "logistica C=1": lambda: LogisticRegression(C=1.0, max_iter=1000),
    "logistica C=0.01": lambda: LogisticRegression(C=0.01, max_iter=1000),
    "bosque sin limite": lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "bosque max_depth=8": lambda: RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    "gradient boosting": lambda: GradientBoostingClassifier(random_state=42),
    "red neuronal": lambda: MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=42),
}

for nombre, crear in MODELOS.items():
    registrar(nombre, crear())

## El registro, ordenado por AUC de prueba

In [ ]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id], order_by=["metrics.auc_prueba DESC"]
).reset_index(drop=True)
columnas = ["run_id", "tags.mlflow.runName", "params.anio_de_corte", "params.huella_datos",
            "metrics.auc_entrenamiento", "metrics.auc_prueba", "metrics.segundos"]
corridas[columnas]

## La corrida final

Elijo la de mayor AUC en prueba. Si dos modelos quedan a menos de 0.005, elegiría el más simple y rápido (la logística antes que un ensamblaje, un ensamblaje antes que la red): la diferencia sería ruido de la división, y el más simple es más fácil de explicar y de servir.

In [ ]:
mejor_run_id = corridas.loc[0, "run_id"]
mlflow.MlflowClient().set_tag(mejor_run_id, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + mejor_run_id + "/modelo")
auc_reproducida = roc_auc_score(y_prueba, modelo_final.predict_proba(X_prueba)[:, 1])
auc_registrada = corridas.loc[0, "metrics.auc_prueba"]
print(f"Final: {corridas.loc[0, 'tags.mlflow.runName']} ({mejor_run_id})")
print(f"AUC registrada {auc_registrada:.6f}   reproducida {auc_reproducida:.6f}   coincide: {abs(auc_reproducida - auc_registrada) < 1e-9}")

In [ ]:
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.auc_prueba DESC"])
corridas.to_csv("corridas_proyecto.csv", index=False)
print("corridas_proyecto.csv:", len(corridas), "corridas")